In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from conch.open_clip_custom import create_model_from_pretrained
from PIL import Image
import pandas as pd
import os
from tqdm import tqdm
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from collections import OrderedDict


/home/linus-dannull/anaconda3/envs/conch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/linus-dannull/anaconda3/envs/conch/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [23]:
# ── Config ────────────────────────────────────────────────────────────────────
model_cfg       = 'conch_ViT-B-16'
checkpoint_path = 'checkpoints/conch/pytorch_model.bin'
image_dir       = 'mhist_data/images'
annotations     = 'mhist_data/annotations.csv'
device          = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Training hyperparameters
BATCH_SIZE      = 32
EPOCHS          = 25
HEAD_LR         = 1e-3      # higher LR for the new randomly-init head
ENCODER_LR      = 1e-5      # very low LR for pretrained transformer blocks
WEIGHT_DECAY    = 1e-4
NUM_UNFROZEN    = 4         # unfreeze the last N transformer blocks, 1 is conservative, 4-6 is aggressive
LABEL_MAP       = {'HP': 0, 'SSA': 1}
EMBED_DIM       = 512
SAVE_PATH       = f'checkpoints/conch/conch_mhist_finetuned_num_unfrozen={NUM_UNFROZEN}.bin'
HEAD_SAVE_PATH  = 'checkpoints/conch/mhist_head.bin'


In [24]:
# ── Dataset ───────────────────────────────────────────────────────────────────
class MHISTDataset(Dataset):
    def __init__(self, df, image_dir, transform):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.image_dir, row['Image Name'])).convert('RGB')
        return self.transform(image), LABEL_MAP[row['Majority Vote Label']]

# ── Model: CONCH visual encoder + classification head ────────────────────────
class CONCHFinetuneClassifier(nn.Module):
    """
    Wraps CONCH's visual encoder and adds a classification head.
    Selectively unfreezes the last N transformer blocks for fine-tuning.
    """
    def __init__(self, conch_model, num_classes=2, embed_dim=512, num_unfrozen_blocks=2):
        super().__init__()
        # CONCH stores the image encoder under .visual
        self.visual = conch_model.visual

        # Classification head
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

        self._configure_trainable_params(num_unfrozen_blocks)

    def _configure_trainable_params(self, num_unfrozen_blocks):
        # Freeze everything in the visual encoder by default
        for p in self.visual.parameters():
            p.requires_grad = False

        # The transformer blocks live at self.visual.trunk.blocks (timm ViT layout)
        # If your CONCH build uses a different attribute path, adjust here.
        try:
            blocks = self.visual.trunk.blocks
        except AttributeError:
            # Fallback for non-timm layouts
            blocks = self.visual.transformer.resblocks

        total_blocks = len(blocks)
        unfreeze_from = max(0, total_blocks - num_unfrozen_blocks)
        print(f"Visual encoder has {total_blocks} blocks; unfreezing blocks "
              f"{unfreeze_from}..{total_blocks - 1}")

        for i, block in enumerate(blocks):
            if i >= unfreeze_from:
                for p in block.parameters():
                    p.requires_grad = True

        # Also unfreeze the final norm layer if present (typical for ViTs)
        for attr in ['norm', 'ln_post', 'trunk.norm']:
            module = self.visual
            for part in attr.split('.'):
                module = getattr(module, part, None)
                if module is None:
                    break
            if module is not None and isinstance(module, nn.Module):
                for p in module.parameters():
                    p.requires_grad = True
                print(f"  also unfroze: visual.{attr}")
                break

    def encode_image(self, x):
        out = self.visual(x)
        # CONCH's visual returns a tuple — typically (pooled, tokens) or similar.
        # The first element is the pooled image embedding we want.
        if isinstance(out, tuple):
            features = out[0]
        else:
            features = out
        return features

    def forward(self, x):
        features = self.encode_image(x)
        features = features / features.norm(dim=-1, keepdim=True)
        return self.head(features)


In [25]:
# ── Load model ────────────────────────────────────────────────────────────────
print("Loading CONCH base model...")
base_model, preprocess = create_model_from_pretrained(model_cfg, checkpoint_path, device=device)

classifier = CONCHFinetuneClassifier(
    base_model,
    num_classes=2,
    embed_dim=EMBED_DIM,
    num_unfrozen_blocks=NUM_UNFROZEN,
).to(device)

# Sanity check on trainable parameters
trainable = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
total     = sum(p.numel() for p in classifier.parameters())
print(f"Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")

# ── Data splits ───────────────────────────────────────────────────────────────
df = pd.read_csv(annotations)
if 'Partition' in df.columns:
    train_df = df[df['Partition'] == 'train']
    test_df  = df[df['Partition'] == 'test']
    print(f"Official split — train: {len(train_df)}, test: {len(test_df)}")
else:
    train_df, test_df = train_test_split(
        df, test_size=0.2, stratify=df['Majority Vote Label'], random_state=42
    )

train_df, val_df = train_test_split(
    train_df, test_size=0.1, stratify=train_df['Majority Vote Label'], random_state=42
)

train_loader = DataLoader(
    MHISTDataset(train_df, image_dir, preprocess),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=4,
    pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    MHISTDataset(val_df, image_dir, preprocess),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=4,
    pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    MHISTDataset(test_df, image_dir, preprocess),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=4,
    pin_memory=torch.cuda.is_available()
)


Loading CONCH base model...
Visual encoder has 12 blocks; unfreezing blocks 8..11
  also unfroze: visual.trunk.norm
Trainable params: 28,419,970 / 90,459,778  (31.42%)
Official split — train: 2175, test: 977


In [26]:
# ── Optimizer with parameter groups (different LRs for head vs encoder) ──────
head_params    = [p for n, p in classifier.named_parameters()
                  if p.requires_grad and n.startswith('head')]
encoder_params = [p for n, p in classifier.named_parameters()
                  if p.requires_grad and not n.startswith('head')]

print(f"  head params:    {sum(p.numel() for p in head_params):,}")
print(f"  encoder params: {sum(p.numel() for p in encoder_params):,}")

optimizer = optim.AdamW([
    {'params': head_params,    'lr': HEAD_LR},
    {'params': encoder_params, 'lr': ENCODER_LR},
], weight_decay=WEIGHT_DECAY)

# Class-weighted loss to help with HP/SSA imbalance (~3:1)
class_counts = train_df['Majority Vote Label'].value_counts()
weights = torch.tensor([
    1.0 / class_counts['HP'],
    1.0 / class_counts['SSA'],
], dtype=torch.float32).to(device)
weights = weights / weights.sum() * 2  # normalize to mean=1
print(f"Class weights — HP: {weights[0]:.3f}, SSA: {weights[1]:.3f}")

criterion = nn.CrossEntropyLoss(weight=weights)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

  head params:    66,946
  encoder params: 28,353,024
Class weights — HP: 0.579, SSA: 1.421


In [27]:
# ── Training loop ─────────────────────────────────────────────────────────────
best_val_bacc = 0.0

for epoch in range(EPOCHS):
    classifier.train()
    train_loss = 0.0
    train_preds, train_gts = [], []

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = classifier(images)
        loss = criterion(logits, labels)
        loss.backward()
        # Gradient clipping helps when fine-tuning transformer blocks
        torch.nn.utils.clip_grad_norm_(
            [p for p in classifier.parameters() if p.requires_grad], max_norm=1.0
        )
        optimizer.step()

        train_loss += loss.item()
        train_preds.extend(logits.argmax(dim=1).cpu().tolist())
        train_gts.extend(labels.cpu().tolist())

    train_bacc = balanced_accuracy_score(train_gts, train_preds)

    # ── Validation ──
    classifier.eval()
    val_preds, val_gts = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            logits = classifier(images)
            val_preds.extend(logits.argmax(dim=1).cpu().tolist())
            val_gts.extend(labels.tolist())

    val_acc  = accuracy_score(val_gts, val_preds)
    val_bacc = balanced_accuracy_score(val_gts, val_preds)
    print(f"  loss: {train_loss/len(train_loader):.4f} | "
          f"train bacc: {train_bacc:.4f} | val acc: {val_acc:.4f} | val bacc: {val_bacc:.4f}")

    if val_bacc > best_val_bacc:
        best_val_bacc = val_bacc

        # Save full visual encoder weights in the same format CONCH expects
        # so they can be reloaded into a CONCH model via the standard loader
        full_state = base_model.state_dict()
        # Overlay the fine-tuned visual encoder weights
        finetuned_visual = classifier.visual.state_dict()
        for k, v in finetuned_visual.items():
            full_state[f'visual.{k}'] = v.cpu()

        os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
        torch.save(full_state, SAVE_PATH)

        # Save the head separately as well (it's not part of the CONCH format)
        torch.save(classifier.head.state_dict(), HEAD_SAVE_PATH)
        print(f"  ✓ saved fine-tuned model to {SAVE_PATH} (val bacc: {best_val_bacc:.4f})")

    scheduler.step()

Epoch 1/25 [train]: 100%|██████████| 62/62 [15:26<00:00, 14.95s/it]


  loss: 0.4932 | train bacc: 0.7480 | val acc: 0.8211 | val bacc: 0.8365
  ✓ saved fine-tuned model to checkpoints/conch/conch_mhist_finetuned_num_unfrozen=4.bin (val bacc: 0.8365)


Epoch 2/25 [train]: 100%|██████████| 62/62 [15:14<00:00, 14.75s/it]


  loss: 0.3421 | train bacc: 0.8441 | val acc: 0.8440 | val bacc: 0.8621
  ✓ saved fine-tuned model to checkpoints/conch/conch_mhist_finetuned_num_unfrozen=4.bin (val bacc: 0.8621)


Epoch 3/25 [train]: 100%|██████████| 62/62 [15:04<00:00, 14.58s/it]


...

  loss: 0.1133 | train bacc: 0.9610 | val acc: 0.8899 | val bacc: 0.8566


Epoch 7/25 [train]: 100%|██████████| 62/62 [8:55:40<00:00, 518.39s/it]     


  loss: 0.0694 | train bacc: 0.9696 | val acc: 0.8761 | val bacc: 0.8234


Epoch 8/25 [train]:   0%|          | 0/62 [00:15<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# ── Test evaluation with best checkpoint ──────────────────────────────────────
print("\n" + "=" * 60)
print("Loading best checkpoint and evaluating on test set...")

# Re-load both pieces
base_model_test, _ = create_model_from_pretrained(model_cfg, SAVE_PATH, device=device)
test_classifier = CONCHFinetuneClassifier(
    base_model_test, num_classes=2, embed_dim=EMBED_DIM, num_unfrozen_blocks=NUM_UNFROZEN
).to(device)
test_classifier.head.load_state_dict(torch.load(HEAD_SAVE_PATH))
test_classifier.eval()

inv_label = {v: k for k, v in LABEL_MAP.items()}
test_preds, test_gts = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Test eval"):
        images = images.to(device)
        logits = test_classifier(images)
        preds = logits.argmax(dim=1).cpu().tolist()
        test_preds.extend([inv_label[p] for p in preds])
        test_gts.extend([inv_label[l] for l in labels.tolist()])

print("\nTest Accuracy:         ", accuracy_score(test_gts, test_preds))
print("Test Balanced Accuracy:", balanced_accuracy_score(test_gts, test_preds))
print("\nClassification report:")
print(classification_report(test_gts, test_preds, digits=4))

print(f"\nFine-tuned CONCH weights saved to: {SAVE_PATH}")
print(f"Classification head saved to:      {HEAD_SAVE_PATH}")
print(f"\nTo use the fine-tuned model for caption generation:")
print(f"  model, preprocess = create_model_from_pretrained(")
print(f"      '{model_cfg}', '{SAVE_PATH}', device=device)")